# Sistema de Recomendação em Produção — sc_base_01

Este notebook documenta o **caminho de produção (serving)** do sistema de recomendação.

O modelo em produção é um **NeuMF** (*Neural Collaborative Filtering*, GMF + MLP — He
et al., 2017) treinado com objetivo de **ranking** e combinado com um **prior de
popularidade** num recomendador **híbrido popularity-first**:

```
score = z(log1p(popularidade)) + alpha * z(NeuMF)
```

O treino e o registro do modelo pertencem ao pipeline MLOps
(`src/scripts/otimizacao_modelo.py`, orquestrado pelo DVC). Aqui **consumimos os
artefatos versionados** em `models/` e demonstramos o que um sistema em produção
precisa garantir:

1. Deploy determinístico a partir dos artefatos salvos (modelo + encoders + config);
2. Avaliação de **ranking** no protocolo *leave-last-out* contra baselines honestos;
3. Geração do **top-7** personalizado, excluindo o que o cliente já comprou;
4. Tratamento de ***cold start*** (cliente fora da base de treino).

## 1. Configuração do ambiente

In [ ]:
import sys
from pathlib import Path

# Localiza a raiz do projeto (onde está o pyproject.toml) para importar `src.*`.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import joblib
import keras
import numpy as np
import pandas as pd
import tensorflow as tf

from src.config.model_utils import (
    TOP_K,
    build_item_catalog,
    evaluate_hybrid,
    evaluate_most_popular,
    evaluate_random_baseline,
    evaluate_ranking,
    recommend_products,
    train_validation_test_split,
)
from src.config.paths import DADOS_TRATADOS, MODELS_DIR

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("Projeto:", ROOT)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

## 2. Carregamento dos artefatos de produção

Um deploy precisa ser **reprodutível**: carregamos exatamente os artefatos produzidos
pelo pipeline — o modelo `.keras`, os quatro `LabelEncoder` e a configuração do
híbrido (`alpha` + vetor de popularidade). Nada é retreinado aqui.

In [ ]:
dados = pd.read_parquet(DADOS_TRATADOS)
encoders = {
    nome: joblib.load(MODELS_DIR / f"{nome}_encoder_otimizado.pkl")
    for nome in ["customer", "product", "category", "subcategory"]
}
modelo = tf.keras.models.load_model(MODELS_DIR / "best_model_recomendacao_otimizado.keras")
config_hibrido = joblib.load(MODELS_DIR / "hybrid_config_otimizado.pkl")

alpha = float(config_hibrido["alpha"])
prior_popularidade = config_hibrido["popularity"]
num_produtos = len(encoders["product"].classes_)
catalogo = build_item_catalog(
    dados, encoders["product"], encoders["category"], encoders["subcategory"]
)

train_pairs, val_pairs, test_pairs = train_validation_test_split(
    dados, encoders["customer"], encoders["product"]
)

print("Interações:", dados.shape)
print(f"Clientes: {len(encoders['customer'].classes_)} | Produtos: {num_produtos} "
      f"| Categorias: {len(encoders['category'].classes_)}")
print(f"Pares -> treino: {len(train_pairs)} | validação: {len(val_pairs)} | teste: {len(test_pairs)}")
print(f"alpha do híbrido (produção): {alpha}")

## 3. Avaliação de ranking (*leave-last-out*)

Protocolo padrão em filtragem colaborativa implícita: o **último** item de cada
cliente é o teste, o **penúltimo** é a validação e o restante é o treino. Para cada
cliente ranqueamos o **catálogo completo**, **excluímos os itens já vistos** e
comparamos o top-k com o item reservado (*held-out*).

Comparamos quatro políticas:

| Política | O que é |
|---|---|
| **Híbrido (produção)** | `z(log1p(pop)) + alpha * z(NeuMF)` com o `alpha` escolhido na validação |
| **NeuMF (neural puro)** | apenas o sinal personalizado da rede |
| **Mais populares** | baseline não personalizado (ranking determinístico) |
| **Aleatório** | esperança analítica do sorteio uniforme |

In [ ]:
metricas = {
    "Híbrido (produção)": evaluate_hybrid(
        modelo, test_pairs, train_pairs, catalogo, prior_popularidade, alpha, k=TOP_K
    ),
    "NeuMF (neural puro)": evaluate_ranking(modelo, test_pairs, train_pairs, catalogo, k=TOP_K),
    "Mais populares": evaluate_most_popular(test_pairs, train_pairs, num_produtos, k=TOP_K),
    "Aleatório (esperança)": evaluate_random_baseline(test_pairs, train_pairs, num_produtos, k=TOP_K),
}

linhas = []
for nome, m in metricas.items():
    linhas.append({
        "Política": nome,
        f"NDCG@{TOP_K}": m.get(f"ndcg@{TOP_K}", np.nan),
        f"MAP@{TOP_K}": m.get(f"map@{TOP_K}", np.nan),
        f"HitRate@{TOP_K}": m.get(f"hit_rate@{TOP_K}", np.nan),
        f"Precision@{TOP_K}": m.get(f"precision@{TOP_K}", np.nan),
        f"Recall@{TOP_K}": m.get(f"recall@{TOP_K}", np.nan),
        f"F1@{TOP_K}": m.get(f"f1@{TOP_K}", np.nan),
    })

tabela = pd.DataFrame(linhas).set_index("Política").round(4).fillna("—")
tabela

### Leitura honesta dos resultados

Nesta base (~12 itens distintos por cliente em um catálogo de ~1.900), o baseline
**mais-populares** é muito forte. O `alpha` é escolhido na validação por uma política
**conservadora** (só adota personalização se ela superar o prior por margem mínima);
quando isso não acontece, o híbrido **degenera explicitamente em `alpha=0`** e
reproduz o baseline popular — em vez de forçar um ganho que não existe. É essa
honestidade que torna as métricas auditáveis.

Os traços (**—**) marcam métricas sem forma fechada para o baseline **aleatório**:
reportamos apenas a esperança **analítica exata** de Hit-Rate, Precision e Recall, em
vez de aproximar NDCG/MAP/F1 por simulação.

In [ ]:
base = metricas["Mais populares"][f"ndcg@{TOP_K}"]
ganho = metricas["Híbrido (produção)"][f"ndcg@{TOP_K}"] / base - 1
print(f"NDCG@{TOP_K} do híbrido sobre o baseline 'mais populares': {ganho:+.1%}")
print(f"NDCG@{TOP_K} do NeuMF puro: {metricas['NeuMF (neural puro)'][f'ndcg@{TOP_K}']:.4f}")
print(f"Hit-Rate@{TOP_K} do aleatório (referência baixa): "
      f"{metricas['Aleatório (esperança)'][f'hit_rate@{TOP_K}']:.4f}")
print()
print("Observação: com alpha=0 o híbrido é, por construção, idêntico ao baseline popular.")

## 4. Serviço de recomendação (top-7)

A função `recommend_products` encapsula o serving: pontua o catálogo, aplica o score
híbrido, remove o que o cliente já comprou e devolve o top-k com metadados.

In [ ]:
CLIENTE = dados["Customer Name"].iloc[0]  # cliente de exemplo (sem nome fixo no código)

historico = (
    dados[dados["Customer Name"] == CLIENTE]
    .sort_values("Order Date")
    .tail(5)[["Order Date", "Product ID", "Product Name", "Category", "Sub-Category"]]
)
print(f"Últimas compras de {CLIENTE}:")
display(historico)

recomendacoes = recommend_products(
    modelo, CLIENTE, dados, encoders["customer"], encoders["product"], catalogo,
    top_k=TOP_K, popularity=prior_popularidade, alpha=alpha,
)
recomendacoes

## 5. *Cold start* — cliente fora da base de treino

Um cliente novo não tem histórico nem embedding. O sistema precisa degradar com
elegância: cai no baseline **mais populares** (nunca em erro).

In [ ]:
cliente_novo = "Cliente Novo (sem histórico)"
try:
    encoders["customer"].transform([cliente_novo])
    print("Cliente encontrado na base.")
except ValueError:
    print(f"'{cliente_novo}' não está na base -> acionando o fallback de cold start.")

recomendacoes_novo = recommend_products(
    modelo, cliente_novo, dados, encoders["customer"], encoders["product"], catalogo,
    top_k=TOP_K, popularity=prior_popularidade, alpha=alpha,
)
recomendacoes_novo

## 6. Conclusão

- O sistema é **reprodutível**: o deploy consome apenas artefatos versionados (DVC),
  sem re-treino.
- A avaliação usa o protocolo **leave-last-out** com **baselines honestos**; quando a
  personalização não é vantajosa, o híbrido assume o prior de popularidade (`alpha=0`).
- O *cold start* é tratado com *fallback* para os itens mais populares.

Próximos passos operacionais: expor `recommend_products` como serviço (API/worker),
monitorar *drift* do catálogo e reavaliar o `alpha` periodicamente na validação.